In [1]:
from pathlib import Path
from typing import Any
import math
import numbers
import pandas as pd

# ============================================================
# RETRIEVE METRIC DEFINITIONS
# ============================================================

InputFilePath = Path(
    r"C:\Users\StevenFoster\APM US\Data and Insights - Documents\Power BI Source Files"
    r"\Balanced Scorecard\ARG Operational Excellence Metric Thresholds.xlsx"
)

MetricsDf = pd.read_excel(InputFilePath, engine="openpyxl")
Metrics = MetricsDf.to_dict(orient="records")

# ============================================================
# CONFIGURATION
# ============================================================

LayoutTableName = "ARG Operational Metric Layout"
OutputFileName = "ARG_Operational_Metric_DAX.txt"
CopyToClipboard = True
GenerateLayoutTable = True
BlankStatusWhenValueIsBlank = True
BaseIndent = 0

# ============================================================
# VALIDATION SETTINGS
# ============================================================

ValidOperators = {"<", "<=", ">", ">=", "=", "<>"}
RequiredFields = {
    "SortOrder",
    "MetricName",
    "MetricKey",
    "FormatString",
    "ThresholdFlag",
    "RedOperator",
    "RedThreshold",
    "AmberOperator",
    "AmberThreshold",
    "YTDFlag",
    "YoYFlag",
    "YTDTargetFlag",
    "MonthTargetFlag",
    "Measure",
    "Target",
}

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def HasTextValue(Value: Any) -> bool:
    """Returns True when a metadata value is not blank or NaN."""
    return pd.notna(Value) and bool(str(Value).strip())


def NormalizeWholeNumber(Value: Any, FieldName: str, MetricName: str) -> int:
    """Converts Excel numeric values or numeric strings to a whole number."""
    if isinstance(Value, bool) or not HasTextValue(Value):
        raise TypeError(
            f"{FieldName} for '{MetricName}' must be a whole number."
        )

    try:
        NumericValue = float(str(Value).strip())
    except (TypeError, ValueError) as ConversionError:
        raise TypeError(
            f"{FieldName} for '{MetricName}' must be numeric. "
            f"Received: {Value!r}."
        ) from ConversionError

    if not math.isfinite(NumericValue) or not NumericValue.is_integer():
        raise TypeError(
            f"{FieldName} for '{MetricName}' must be a whole number. "
            f"Received: {Value!r}."
        )

    return int(NumericValue)


def NormalizeNumber(Value: Any, FieldName: str, MetricName: str) -> int | float:
    """Converts Excel numeric values or numeric strings to a finite number."""
    if isinstance(Value, bool) or not HasTextValue(Value):
        raise TypeError(
            f"{FieldName} for '{MetricName}' must be numeric."
        )

    try:
        NumericValue = float(str(Value).strip())
    except (TypeError, ValueError) as ConversionError:
        raise TypeError(
            f"{FieldName} for '{MetricName}' must be numeric. "
            f"Received: {Value!r}."
        ) from ConversionError

    if not math.isfinite(NumericValue):
        raise TypeError(
            f"{FieldName} for '{MetricName}' must be a finite number."
        )

    return int(NumericValue) if NumericValue.is_integer() else NumericValue


def ValidateMetrics(MetricList: list[dict[str, Any]]) -> None:
    """Validates and normalizes metric metadata before generating DAX."""
    if not MetricList:
        raise ValueError("Metrics cannot be empty.")

    MetricKeys: list[str] = []
    SortOrders: list[int] = []

    for MetricIndex, Metric in enumerate(MetricList, start=1):
        MissingFields = RequiredFields.difference(Metric.keys())

        if MissingFields:
            MissingText = ", ".join(sorted(MissingFields))
            raise ValueError(
                f"Metric {MetricIndex} is missing required fields: {MissingText}"
            )

        MetricName = str(Metric["MetricName"]).strip()
        MetricKey = str(Metric["MetricKey"]).strip()
        FormatString = str(Metric["FormatString"]).strip()

        if not MetricName:
            raise ValueError(f"Metric {MetricIndex} has a blank MetricName.")

        if not MetricKey:
            raise ValueError(f"Metric {MetricIndex} has a blank MetricKey.")

        if not MetricKey.replace("_", "").isalnum():
            raise ValueError(
                f"MetricKey '{MetricKey}' contains unsupported characters. "
                "Use letters, numbers, or underscores only."
            )

        if not FormatString:
            raise ValueError(f"Metric '{MetricName}' has a blank FormatString.")

        for FlagName in (
            "ThresholdFlag",
            "YTDFlag",
            "YoYFlag",
            "YTDTargetFlag",
            "MonthTargetFlag",
        ):
            FlagValue = str(Metric[FlagName]).strip().lower()
            if FlagValue not in {"yes", "no"}:
                raise ValueError(
                    f"Metric '{MetricName}' has an invalid {FlagName}. "
                    "Use Yes or No."
                )
            Metric[FlagName] = FlagValue

        ThresholdFlag = Metric["ThresholdFlag"]
        RedOperator = str(Metric["RedOperator"]).strip()
        AmberOperator = str(Metric["AmberOperator"]).strip()

        if ThresholdFlag == "yes":
            if RedOperator not in ValidOperators:
                raise ValueError(
                    f"Metric '{MetricName}' has unsupported RedOperator "
                    f"'{RedOperator}'."
                )

            if AmberOperator not in ValidOperators:
                raise ValueError(
                    f"Metric '{MetricName}' has unsupported AmberOperator "
                    f"'{AmberOperator}'."
                )

            Metric["RedThreshold"] = NormalizeNumber(
                Metric["RedThreshold"], "RedThreshold", MetricName
            )
            Metric["AmberThreshold"] = NormalizeNumber(
                Metric["AmberThreshold"], "AmberThreshold", MetricName
            )

        Metric["SortOrder"] = NormalizeWholeNumber(
            Metric["SortOrder"], "SortOrder", MetricName
        )

        if Metric["YTDTargetFlag"] == "yes" and not HasTextValue(Metric["Target"]):
            raise ValueError(
                f"Metric '{MetricName}' has YTDTargetFlag set to Yes "
                "but Target is blank."
            )

        if Metric["MonthTargetFlag"] == "yes" and not HasTextValue(Metric["Target"]):
            raise ValueError(
                f"Metric '{MetricName}' has MonthTargetFlag set to Yes "
                "but Target is blank."
            )

        MetricKeys.append(MetricKey)
        SortOrders.append(Metric["SortOrder"])

    DuplicateMetricKeys = {
        MetricKey for MetricKey in MetricKeys if MetricKeys.count(MetricKey) > 1
    }
    if DuplicateMetricKeys:
        DuplicateText = ", ".join(sorted(DuplicateMetricKeys))
        raise ValueError(f"Duplicate MetricKey values found: {DuplicateText}")

    DuplicateSortOrders = {
        SortOrder for SortOrder in SortOrders if SortOrders.count(SortOrder) > 1
    }
    if DuplicateSortOrders:
        DuplicateText = ", ".join(
            str(SortOrder) for SortOrder in sorted(DuplicateSortOrders)
        )
        raise ValueError(f"Duplicate SortOrder values found: {DuplicateText}")


def GetVariableName(MetricKey: str) -> str:
    """Converts a metadata key to the requested Power BI variable pattern."""
    CleanMetricKey = str(MetricKey).strip()
    if not CleanMetricKey:
        raise ValueError("A variable name cannot be blank.")
    return CleanMetricKey[0].lower() + CleanMetricKey[1:]


def GetMeasureReference(MeasureName: Any) -> str:
    """Returns a DAX measure reference, adding brackets when needed."""
    CleanMeasureName = str(MeasureName).strip()
    if not CleanMeasureName:
        raise ValueError("A measure name cannot be blank.")
    if CleanMeasureName.startswith("[") and CleanMeasureName.endswith("]"):
        return CleanMeasureName
    return f"[{CleanMeasureName}]"


def EscapeDaxText(Value: Any) -> str:
    return str(Value).replace('"', '""')


def FormatDaxNumber(Value: int | float) -> str:
    if isinstance(Value, bool) or not isinstance(Value, numbers.Real):
        raise TypeError("Threshold values must be numeric.")
    return format(Value, ".15g")


def GetIndentedText(Text: str, IndentLevel: int) -> str:
    Prefix = " " * IndentLevel
    return "\n".join(
        f"{Prefix}{Line}" if Line else "" for Line in Text.splitlines()
    )


def BuildSectionHeader(SectionName: str) -> str:
    return (
        "/* ============================================================\n"
        f"   {SectionName}\n"
        "   ============================================================ */"
    )

# ============================================================
# DAX LAYOUT TABLE GENERATOR
# ============================================================

def BuildLayoutTable(TableName: str, MetricList: list[dict[str, Any]]) -> str:
    SortedMetrics = sorted(MetricList, key=lambda Metric: Metric["SortOrder"])
    TableRows = []

    for Metric in SortedMetrics:
        MetricName = EscapeDaxText(Metric["MetricName"])
        MetricKey = EscapeDaxText(Metric["MetricKey"])
        SortOrder = Metric["SortOrder"]
        TableRows.append(f'    {{{SortOrder}, "{MetricName}", "{MetricKey}"}}')

    RowText = ",\n".join(TableRows)
    return (
        f"{TableName} =\n"
        "DATATABLE(\n"
        '    "SortOrder", INTEGER,\n'
        '    "MetricName", STRING,\n'
        '    "MetricKey", STRING,\n'
        "{\n"
        f"{RowText}\n"
        "}\n"
        ")"
    )

# ============================================================
# PLACEHOLDER VARIABLE GENERATOR
# ============================================================

def BuildPlaceholderVariables(MetricList: list[dict[str, Any]]) -> str:
    PlaceholderBlocks = [BuildSectionHeader("OPERATIONAL EXCELLENCE PLACEHOLDERS")]

    for Metric in MetricList:
        VariableName = GetVariableName(Metric["MetricKey"])
        MeasureReference = GetMeasureReference(Metric["Measure"])
        TargetReference = (
            GetMeasureReference(Metric["Target"])
            if HasTextValue(Metric["Target"])
            else ""
        )

        YtdFlag = Metric["YTDFlag"] == "yes"
        YoyFlag = Metric["YoYFlag"] == "yes"
        YtdTargetFlag = Metric["YTDTargetFlag"] == "yes"
        MonthTargetFlag = Metric["MonthTargetFlag"] == "yes"

        VariableLines = [
            (
                f"VAR _{VariableName}Month3 = "
                f"CALCULATE({MeasureReference}, ALL(dimdate), "
                f"dimdate[Month_Year] = _month3)"
            ),
            (
                f"VAR _{VariableName}Month2 = "
                f"CALCULATE({MeasureReference}, ALL(dimdate), "
                f"dimdate[Month_Year] = _month2)"
            ),
            (
                f"VAR _{VariableName}Month1 = "
                f"CALCULATE({MeasureReference}, ALL(dimdate), "
                f"dimdate[Month_Year] = _month1)"
            ),
        ]

        if YtdFlag:
            VariableLines.append(
                f"VAR _{VariableName}Ytd = "
                f"CALCULATE({MeasureReference}, ALL(dimdate), "
                f"dimdate[Fiscal Year] = _year, "
                f"dimdate[Month Sort] <= _currentmonthsort)"
            )

        if YtdTargetFlag:
            VariableLines.append(
                f"VAR _{VariableName}Target = "
                f"CALCULATE({TargetReference}, ALL(dimdate), "
                f"dimdate[Fiscal Year] = _year, "
                f"dimdate[Month Sort] <= _currentmonthsort)"
            )

        if YtdFlag and YtdTargetFlag:
            VariableLines.append(
                f"VAR _{VariableName}Display = "
                f'FORMAT(_{VariableName}Ytd, "#,##") '
                f'& IF(ISBLANK(DIVIDE(_{VariableName}Ytd, _{VariableName}Target)), '
                f'"", " (" & FORMAT(DIVIDE(_{VariableName}Ytd, '
                f'_{VariableName}Target), "0%") & ")")'
            )

        if YoyFlag:
            if not YtdFlag:
                raise ValueError(
                    f"Metric '{Metric['MetricName']}' has YoYFlag set to Yes "
                    "but YTDFlag is No."
                )
            VariableLines.extend(
                [
                    (
                        f"VAR _{VariableName}Pytd = "
                        f"CALCULATE({MeasureReference}, ALL(dimdate), "
                        f"dimdate[Fiscal Year] = _year - 1, "
                        f"dimdate[Date] <= _priorYear)"
                    ),
                    f"VAR _{VariableName}Var = _{VariableName}Ytd - _{VariableName}Pytd",
                    (
                        f"VAR _{VariableName}VariancePct = "
                        f"DIVIDE(_{VariableName}Var, _{VariableName}Pytd)"
                    ),
                    (
                        f"VAR _{VariableName}Trend = "
                        f"SWITCH(TRUE(), ISBLANK(_{VariableName}Var), BLANK(), "
                        f'_{VariableName}Var > 0, "up", '
                        f'_{VariableName}Var < 0, "down", BLANK())'
                    ),
                ]
            )

        if MonthTargetFlag:
            MetricFormatString = EscapeDaxText(Metric["FormatString"])
            VariableLines.extend(
                [
                    (
                        f"VAR _{VariableName}TargetMonth3 = "
                        f"CALCULATE({TargetReference}, ALL(dimdate), "
                        f"dimdate[Month_Year] = _month3)"
                    ),
                    (
                        f"VAR _{VariableName}TargetMonth2 = "
                        f"CALCULATE({TargetReference}, ALL(dimdate), "
                        f"dimdate[Month_Year] = _month2)"
                    ),
                    (
                        f"VAR _{VariableName}TargetMonth1 = "
                        f"CALCULATE({TargetReference}, ALL(dimdate), "
                        f"dimdate[Month_Year] = _month1)"
                    ),
                    (
                        f"VAR _{VariableName}Month3Display = "
                        f'FORMAT(DIVIDE(_{VariableName}Month3,100), "{MetricFormatString}") '
                        f'& IF(ISBLANK(DIVIDE(_{VariableName}Month3, '
                        f'_{VariableName}TargetMonth3)), "", " (" & '
                        f'FORMAT(DIVIDE(_{VariableName}Month3, '
                        f'_{VariableName}TargetMonth3), "0%") & ")")'
                    ),
                    (
                        f"VAR _{VariableName}Month2Display = "
                        f'FORMAT(DIVIDE(_{VariableName}Month2,100), "{MetricFormatString}") '
                        f'& IF(ISBLANK(DIVIDE(_{VariableName}Month2, '
                        f'_{VariableName}TargetMonth2)), "", " (" & '
                        f'FORMAT(DIVIDE(_{VariableName}Month2, '
                        f'_{VariableName}TargetMonth2), "0%") & ")")'
                    ),
                    (
                        f"VAR _{VariableName}Month1Display = "
                        f'FORMAT(DIVIDE(_{VariableName}Month1,100), "{MetricFormatString}") '
                        f'& IF(ISBLANK(DIVIDE(_{VariableName}Month1, '
                        f'_{VariableName}TargetMonth1)), "", " (" & '
                        f'FORMAT(DIVIDE(_{VariableName}Month1, '
                        f'_{VariableName}TargetMonth1), "0%") & ")")'
                    ),
                ]
            )

        PlaceholderBlocks.append("\n".join(VariableLines))

    return "\n\n".join(PlaceholderBlocks)

# ============================================================
# STATUS VARIABLE GENERATOR
# ============================================================

def BuildStatusVariables(
    MetricList: list[dict[str, Any]], ReturnBlankStatus: bool
) -> str:
    StatusBlocks = [BuildSectionHeader("OPERATIONAL EXCELLENCE STATUS")]

    for Metric in MetricList:
        VariableName = GetVariableName(Metric["MetricKey"])

        if Metric["ThresholdFlag"] == "no":
            StatusBlocks.append(f"VAR _{VariableName}Status = BLANK()")
            continue

        RedOperator = str(Metric["RedOperator"]).strip()
        RedThreshold = FormatDaxNumber(Metric["RedThreshold"])
        AmberOperator = str(Metric["AmberOperator"]).strip()
        AmberThreshold = FormatDaxNumber(Metric["AmberThreshold"])

        StatusLines = [
            f"VAR _{VariableName}Status =",
            "    SWITCH(",
            "        TRUE(),",
        ]

        if ReturnBlankStatus:
            StatusLines.append(
                f"        ISBLANK(_{VariableName}Month1), BLANK(),"
            )

        StatusLines.extend(
            [
                f'        _{VariableName}Month1 {RedOperator} {RedThreshold}, "red",',
                f'        _{VariableName}Month1 {AmberOperator} {AmberThreshold}, "amber",',
                '        "green"',
                "    )",
            ]
        )
        StatusBlocks.append("\n".join(StatusLines))

    return "\n\n".join(StatusBlocks)

# ============================================================
# SWITCH GENERATORS
# ============================================================

def BuildMetricSwitch(
    VariableName: str,
    MetricList: list[dict[str, Any]],
    ValueSuffix: str | None = None,
    ValueSuffixField: str | None = None,
    IndentLevel: int = 4,
) -> str:
    if ValueSuffix is None and ValueSuffixField is None:
        raise ValueError("ValueSuffix or ValueSuffixField must be provided.")

    Indent = " " * IndentLevel
    InnerIndent = " " * (IndentLevel + 4)
    ValueIndent = " " * (IndentLevel + 8)
    SwitchLines = [
        f"{Indent}VAR _{VariableName} =",
        f"{InnerIndent}SWITCH(",
        f"{ValueIndent}_metricKey,",
    ]

    MetricLines = []
    for Metric in MetricList:
        MetricKeyText = EscapeDaxText(Metric["MetricKey"])
        MetricVariable = GetVariableName(Metric["MetricKey"])
        Suffix = Metric.get(ValueSuffixField) if ValueSuffixField else ValueSuffix

        if not HasTextValue(Suffix):
            raise ValueError(
                f"Metric '{Metric['MetricName']}' does not have a valid "
                f"suffix for {VariableName}."
            )

        MetricLines.append(
            f'{ValueIndent}"{MetricKeyText}", _{MetricVariable}{Suffix}'
        )

    for MetricIndex, MetricLine in enumerate(MetricLines):
        LineEnding = "" if MetricIndex == len(MetricLines) - 1 else ","
        SwitchLines.append(f"{MetricLine}{LineEnding}")

    SwitchLines.append(f"{InnerIndent})")
    return "\n".join(SwitchLines)


def BuildFormatSwitch(
    MetricList: list[dict[str, Any]], IndentLevel: int = 4
) -> str:
    Indent = " " * IndentLevel
    InnerIndent = " " * (IndentLevel + 4)
    ValueIndent = " " * (IndentLevel + 8)
    SwitchLines = [
        f"{Indent}VAR _formatString =",
        f"{InnerIndent}SWITCH(",
        f"{ValueIndent}_metricKey,",
    ]

    MetricLines = []
    for Metric in MetricList:
        MetricKey = EscapeDaxText(Metric["MetricKey"])
        FormatString = EscapeDaxText(Metric["FormatString"])
        MetricLines.append(f'{ValueIndent}"{MetricKey}", "{FormatString}"')

    for MetricIndex, MetricLine in enumerate(MetricLines):
        LineEnding = "" if MetricIndex == len(MetricLines) - 1 else ","
        SwitchLines.append(f"{MetricLine}{LineEnding}")

    SwitchLines.append(f"{InnerIndent})")
    return "\n".join(SwitchLines)

# ============================================================
# DYNAMIC HTML ROW GENERATOR
# ============================================================

def BuildOperationalRows(
    TableName: str, MetricList: list[dict[str, Any]]
) -> str:
    StatusSwitch = BuildMetricSwitch(
        VariableName="status",
        MetricList=MetricList,
        ValueSuffix="Status",
        IndentLevel=4,
    )

    YtdMetrics = [Metric for Metric in MetricList if Metric["YTDFlag"] == "yes"]
    for Metric in YtdMetrics:
        Metric["YtdSuffix"] = (
            "Display" if Metric["YTDTargetFlag"] == "yes" else "Ytd"
        )

    YtdSwitch = BuildMetricSwitch(
        VariableName="ytd",
        MetricList=YtdMetrics,
        ValueSuffixField="YtdSuffix",
        IndentLevel=4,
    )

    for Metric in MetricList:
        HasMonthTarget = Metric["MonthTargetFlag"] == "yes"
        Metric["Month3Suffix"] = "Month3Display" if HasMonthTarget else "Month3"
        Metric["Month2Suffix"] = "Month2Display" if HasMonthTarget else "Month2"
        Metric["Month1Suffix"] = "Month1Display" if HasMonthTarget else "Month1"

    MonthPrior2Switch = BuildMetricSwitch(
        VariableName="monthPrior2",
        MetricList=MetricList,
        ValueSuffixField="Month3Suffix",
        IndentLevel=4,
    )
    MonthPrior1Switch = BuildMetricSwitch(
        VariableName="monthPrior1",
        MetricList=MetricList,
        ValueSuffixField="Month2Suffix",
        IndentLevel=4,
    )
    CurrentMonthSwitch = BuildMetricSwitch(
        VariableName="currentMonth",
        MetricList=MetricList,
        ValueSuffixField="Month1Suffix",
        IndentLevel=4,
    )

    YoyMetrics = [Metric for Metric in MetricList if Metric["YoYFlag"] == "yes"]
    TrendSwitch = ""
    VarianceSwitch = ""

    if YoyMetrics:
        TrendSwitch = BuildMetricSwitch(
            VariableName="trendIcon",
            MetricList=YoyMetrics,
            ValueSuffix="Trend",
            IndentLevel=4,
        )
        VarianceSwitch = BuildMetricSwitch(
            VariableName="pyVAR",
            MetricList=YoyMetrics,
            ValueSuffix="VariancePct",
            IndentLevel=4,
        )

    FormatSwitch = BuildFormatSwitch(MetricList=MetricList, IndentLevel=4)

    TrendHtml = "        VAR _trendHtml = \"\"\n"
    if YoyMetrics:
        TrendHtml = (
            "        VAR _trendHtml =\n"
            "            SWITCH(\n"
            "                TRUE(),\n"
            "                ISBLANK(_pyVAR), \"\",\n"
            "                _trendIcon = \"up\",\n"
            "                    \"<span class='trend up'>▲ \"\n"
            "                        & FORMAT(_pyVAR, \"0.0%\")\n"
            "                        & \"</span>\",\n"
            "                _trendIcon = \"down\",\n"
            "                    \"<span class='trend down'>▼ \"\n"
            "                        & FORMAT(ABS(_pyVAR), \"0.0%\")\n"
            "                        & \"</span>\",\n"
            "                FORMAT(_pyVAR, \"0.0%\")\n"
            "            )"
        )

    return (
        f"{BuildSectionHeader('DYNAMIC HTML CREATION')}\n"
        "VAR _operationalRows =\n"
        "    CONCATENATEX(\n"
        f"        '{EscapeDaxText(TableName)}',\n"
        "        VAR _metricKey = [MetricKey]\n\n"
        f"{StatusSwitch}\n\n"
        f"{YtdSwitch}\n\n"
        f"{MonthPrior2Switch}\n\n"
        f"{MonthPrior1Switch}\n\n"
        f"{CurrentMonthSwitch}\n\n"
        f"{TrendSwitch}\n\n"
        f"{VarianceSwitch}\n\n"
        f"{TrendHtml}\n\n"
        f"{FormatSwitch}\n\n"
        "        VAR _ytdHtml =\n"
        "            IF(\n"
        "                ISBLANK(_ytd),\n"
        "                \"\",\n"
        "                    FORMAT(_ytd, _formatString)\n"
        "            )\n"
        "        VAR _monthPrior2Html =\n"
        "            IF(\n"
        "                ISBLANK(_monthPrior2),\n"
        "                \"\",\n"
        "                    FORMAT(_monthPrior2, _formatString)\n"
        "            )\n"
        "        VAR _monthPrior1Html =\n"
        "            IF(\n"
        "                ISBLANK(_monthPrior1),\n"
        "                \"\",\n"
        "                    FORMAT(_monthPrior1, _formatString)\n"
        "            )\n"
        "        VAR _currentMonthHtml =\n"
        "            IF(\n"
        "                ISBLANK(_currentMonth),\n"
        "                \"\",\n"
        "                    FORMAT(_currentMonth, _formatString)\n"
        "            )\n"
        "        RETURN\n"
        "            \"<tr>\"\n"
        "                & \"<td class='label'>\" & [MetricName] & \"</td>\"\n"
        "                & \"<td class='status'><span class='dot \"\n"
        "                & COALESCE(_status, \"\") & \"'></span></td>\"\n"
        "                & \"<td class='number'>\" & _ytdHtml & \"</td>\"\n"
        "                & \"<td class='number'>\" & _monthPrior2Html & \"</td>\"\n"
        "                & \"<td class='number'>\" & _monthPrior1Html & \"</td>\"\n"
        "                & \"<td class='number'>\" & _currentMonthHtml & \"</td>\"\n"
        "                & \"<td class='number'>\" & _trendHtml & \"</td>\"\n"
        "                & \"</tr>\",\n"
        "        \"\",\n"
        "        [SortOrder],\n"
        "        ASC\n"
        "    )\n\n"
        "RETURN\n"
        "    \"\n"
        "    <!-- OPERATIONAL EXCELLENCE -->\n"
        "    <tr>\n"
        "      <td colspan='6' class='header'>\n"
        "        Operational Excellence\n"
        "      </td>\n"
        "    </tr>\n"
        "    <tr>\n"
        "      <td class='left'>\n"
        "        <table class='metric'>\n"
        "          <colgroup>\n"
        "            <col style='width:34%'>\n"
        "            <col style='width:3%'>\n"
        "            <col style='width:12.6%'>\n"
        "            <col style='width:12.6%'>\n"
        "            <col style='width:12.6%'>\n"
        "            <col style='width:12.6%'>\n"
        "            <col style='width:12.6%'>\n"
        "          </colgroup>\n"
        "          <tr class='subheader'>\n"
        "            <th></th>\n"
        "            <th></th>\n"
        "            <th>PY '\" & FORMAT(RIGHT(_year, 2), \"##\") & \" YTD</th>\n"
        "            <th>\" & _displayMonth3 & \"</th>\n"
        "            <th>\" & _displayMonth2 & \"</th>\n"
        "            <th>\" & _displayMonth1 & \"</th>\n"
        "            <th>\" & _trending & \"</th>\n"
        "          </tr>\"\n"
        "        & _operationalRows\n"
        "        & \"\n"
        "        </table>\n"
        "      </td>\n"
        "      <td class='right commentbackground'>\n"
        "        <div class='commentsheader'></div>\n"
        "        <div class='commentary'></div>\n"
        "      </td>\n"
        "    </tr>\n"
        "    \""
    )

# ============================================================
# FULL DAX GENERATOR
# ============================================================

def GenerateDax(
    TableName: str,
    MetricList: list[dict[str, Any]],
    IncludeLayoutTable: bool = True,
    ReturnBlankStatus: bool = True,
    IndentLevel: int = 0,
) -> str:
    ValidateMetrics(MetricList)
    SortedMetrics = sorted(MetricList, key=lambda Metric: Metric["SortOrder"])
    OutputSections = []

    if IncludeLayoutTable:
        OutputSections.extend(
            [
                BuildSectionHeader("METRIC LAYOUT TABLE"),
                BuildLayoutTable(TableName=TableName, MetricList=SortedMetrics),
            ]
        )

    OutputSections.extend(
        [
            BuildSectionHeader("TREND COLUMN VARIABLE"),
            'VAR _trending = "YoY Trend"',
            BuildSectionHeader("TIME PLACEHOLDERS"),
            "VAR _year = [Current Fiscal Year]",
            "VAR _priorYear = [Prior Fiscal YTD]",
            "VAR _currentmonthsort = [Current Month Sort]",
            "VAR _month3 = [Month Year Prior 2]",
            "VAR _month2 = [Month Year Prior 1]",
            "VAR _month1 = [Month Year Current]",
            "VAR _displayMonth3 = [Month Display Prior 2]",
            "VAR _displayMonth2 = [Month Display Prior 1]",
            "VAR _displayMonth1 = [Month Display Current]",
            BuildPlaceholderVariables(MetricList=SortedMetrics),
            BuildStatusVariables(
                MetricList=SortedMetrics,
                ReturnBlankStatus=ReturnBlankStatus,
            ),
            BuildOperationalRows(TableName=TableName, MetricList=SortedMetrics),
        ]
    )

    GeneratedDax = "\n\n".join(OutputSections)
    return GetIndentedText(GeneratedDax.strip(), IndentLevel)

# ============================================================
# CLIPBOARD FUNCTION
# ============================================================

def CopyTextToClipboard(Text: str) -> bool:
    try:
        import pyperclip

        pyperclip.copy(Text)
        if pyperclip.paste() != Text:
            print("Warning: Clipboard verification did not match the generated DAX.")
            return False
        return True
    except ImportError:
        print("Clipboard copy skipped because pyperclip is not installed.")
        print("Install it in Jupyter with: %pip install pyperclip")
        return False
    except Exception as ClipboardError:
        print("Clipboard copy failed, but the DAX file was still created.")
        print(f"Clipboard error: {ClipboardError}")
        return False

# ============================================================
# GENERATE, SAVE, COPY, AND DISPLAY
# ============================================================

try:
    DaxCode = GenerateDax(
        TableName=LayoutTableName,
        MetricList=Metrics,
        IncludeLayoutTable=GenerateLayoutTable,
        ReturnBlankStatus=BlankStatusWhenValueIsBlank,
        IndentLevel=BaseIndent,
    )

    OutputPath = Path(OutputFileName).resolve()
    OutputPath.write_text(DaxCode, encoding="utf-8")

    ClipboardCopied = CopyTextToClipboard(DaxCode) if CopyToClipboard else False

    print("=" * 70)
    print("DAX GENERATION COMPLETE")
    print("=" * 70)
    print(f"Metric count: {len(Metrics)}")
    print(f"Layout table: {LayoutTableName}")
    print(f"Saved file: {OutputPath}")

    if CopyToClipboard:
        print(
            "Clipboard: DAX copied successfully"
            if ClipboardCopied
            else "Clipboard: DAX was not copied"
        )
    else:
        print("Clipboard: Disabled in configuration")

    print("=" * 70)
    print()
    print(DaxCode)

except Exception as GenerationError:
    print("=" * 70)
    print("DAX GENERATION FAILED")
    print("=" * 70)
    print(str(GenerationError))
    raise


DAX GENERATION COMPLETE
Metric count: 5
Layout table: ARG Operational Metric Layout
Saved file: C:\Users\StevenFoster\APM US\Data and Insights - Documents\Python Scripts\Balanced Scorecard\ARG_Operational_Metric_DAX.txt
Clipboard: DAX copied successfully

/* ============================================================
   METRIC LAYOUT TABLE
   ============================================================ */

ARG Operational Metric Layout =
DATATABLE(
    "SortOrder", INTEGER,
    "MetricName", STRING,
    "MetricKey", STRING,
{
    {1, "CO Determinations (% Target)", "_coDeterminations"},
    {2, "CO TAT (Application Review)", "_coTatApplicationReview"},
    {3, "CO Avg Days Ahead of Target", "_coAvgDaysAheadOfTarget"},
    {4, "PA TAT (Application Review)", "_paTatApplicationReview"},
    {5, "PA Avg Days Ahead of Target", "_paAvgDaysAheadOfTarget"}
}
)

/* ============================================================
   TREND COLUMN VARIABLE
   =========================================